In [5]:
# %% [code]
import sys, subprocess, io, os, traceback, warnings

# Suppress all pip/warning noise during install attempts
warnings.filterwarnings("ignore")
os.environ["PIP_NO_INPUT"] = "1"

# ---------------------------
# Silent install helper (suppresses ALL output)
# ---------------------------
def pip_install_silent(pkg: str) -> bool:
    """Try to install package with ALL output suppressed"""
    try:
        with open(os.devnull, 'w') as devnull:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", pkg],
                stdout=devnull, stderr=devnull
            )
        return True
    except Exception:
        return False

# ---------------------------
# Try import, install silently if missing
# ---------------------------
def ensure_package(pkg: str, import_name: str = None):
    """Ensure package is available - install silently if needed"""
    if import_name is None:
        import_name = pkg.replace("-", "_").split('.')[0]
    try:
        __import__(import_name)
        return True
    except ImportError:
        # Try silent install
        if pip_install_silent(pkg):
            try:
                __import__(import_name)
                return True
            except ImportError:
                return False
        return False

# Required packages
packages = [
    ("azure-ai-vision-imageanalysis", "azure.ai.vision.imageanalysis"),
    ("azure-ai-contentsafety", "azure.ai.contentsafety"),
    ("azure-core", "azure.core"),
    ("openai", "openai"),
    ("pillow", "PIL"),
    ("ipywidgets", "ipywidgets"),
]

# Attempt to ensure all packages (silently)
for pkg, imp in packages:
    ensure_package(pkg, imp)

# ---------------------------
# Imports (will work if pre-installed or install succeeded)
# ---------------------------
from IPython.display import display, Image as IPyImage
from ipywidgets import widgets, VBox, HBox, Layout
from PIL import Image

from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
from azure.ai.contentsafety import ContentSafetyClient
from azure.ai.contentsafety.models import AnalyzeTextOptions
from openai import AzureOpenAI


# ---------------------------
# Credentials (replace these)
# ---------------------------
AZURE_OPENAI_ENDPOINT = "https://openaimultimodal.openai.azure.com/"
AZURE_OPENAI_KEY = "8PQzAHTn8Q3eT0vHJitLIa4RPAD5RVzePDE5UirQ2mtWNJ3n1sTLJQQJ99CCACYeBjFXJ3w3AAABACOG5ryT"
AZURE_OPENAI_API_VERSION = "2024-02-15-preview"

AZURE_CV_ENDPOINT = "https://visionmultimodal.cognitiveservices.azure.com/"
AZURE_CV_KEY = "7Li69TAPrcAXP6u6M2kiWw9WOUQP10Gv0wmbdaRLilNYxFDpMuBqJQQJ99CCACYeBjFXJ3w3AAAFACOGV7kb"

AZURE_CONTENT_SAFETY_ENDPOINT = "https://safetymultimodal.cognitiveservices.azure.com/"
AZURE_CONTENT_SAFETY_KEY = "4QU67ydl4N8h3m9locwcDzaVJ1BqKB8GMYUuKFGCGC7NeBcW1ffGJQQJ99CCACYeBjFXJ3w3AAAHACOGevHQ"

# IMPORTANT: must be your Azure OpenAI DEPLOYMENT NAME (not model name)
AZURE_OPENAI_DEPLOYMENT = "Deploygpt-4o"  # change if your deployment name differs


# ---------------------------
# Configure clients
# ---------------------------
cv_client = None
openai_client = None
safety_client = None

if AZURE_CV_KEY:
    try:
        cv_client = ImageAnalysisClient(
            endpoint=AZURE_CV_ENDPOINT,
            credential=AzureKeyCredential(AZURE_CV_KEY)
        )
    except Exception:
        pass

if AZURE_OPENAI_KEY:
    try:
        openai_client = AzureOpenAI(
            api_key=AZURE_OPENAI_KEY,
            azure_endpoint=AZURE_OPENAI_ENDPOINT,
            api_version=AZURE_OPENAI_API_VERSION
        )
    except Exception:
        pass

if AZURE_CONTENT_SAFETY_KEY:
    try:
        safety_client = ContentSafetyClient(
            endpoint=AZURE_CONTENT_SAFETY_ENDPOINT,
            credential=AzureKeyCredential(AZURE_CONTENT_SAFETY_KEY)
        )
    except Exception:
        pass

# ✅ Clean success message (only output shown if all good)
if all([cv_client, openai_client, safety_client]):
    print("✅ Azure services connected successfully")
else:
    missing = []
    if not cv_client: missing.append("Computer Vision")
    if not openai_client: missing.append("OpenAI")
    if not safety_client: missing.append("Content Safety")
    print(f"⚠️  Some services not initialized: {', '.join(missing)}")
    print("   → Check that environment variables are set correctly")


# ---------------------------
# State
# ---------------------------
conversation_history = {}
current_image_bytes = None


# ---------------------------
# Helpers
# ---------------------------
def unwrap_values(x):
    if x is None: return []
    if isinstance(x, (list, tuple)): return list(x)
    if isinstance(x, dict):
        if "values" in x and isinstance(x["values"], (list, tuple)):
            return list(x["values"])
        return []
    if hasattr(x, "values") and not callable(getattr(x, "values")):
        v = getattr(x, "values")
        return list(v) if isinstance(v, (list, tuple)) else []
    try: return list(x)
    except Exception: return []

def safe_name(obj):
    if obj is None: return None
    if isinstance(obj, str): return obj
    if isinstance(obj, dict):
        return obj.get("name") or obj.get("text") or obj.get("label") or str(obj)
    if hasattr(obj, "name"): return getattr(obj, "name")
    if hasattr(obj, "text"): return getattr(obj, "text")
    return str(obj)

def safe_conf(obj):
    if obj is None: return None
    if isinstance(obj, dict): return obj.get("confidence")
    if hasattr(obj, "confidence"):
        c = getattr(obj, "confidence")
        return c if isinstance(c, (int, float)) else None
    return None

def extract_objects(result, limit=10):
    objs_container = getattr(result, "objects", None)
    objs = unwrap_values(getattr(objs_container, "list", None)) if objs_container else []
    if not objs: return ["None detected"]
    out = [safe_name(o) for o in objs[:limit] if safe_name(o)]
    return out if out else ["None detected"]

def extract_tags(result, limit=5):
    tags_container = getattr(result, "tags", None)
    tags = unwrap_values(getattr(tags_container, "list", None)) if tags_container else []
    if not tags: return ["None detected"]
    out = []
    for t in tags[:limit]:
        nm, cf = safe_name(t), safe_conf(t)
        if nm and isinstance(cf, (int, float)):
            out.append(f"{nm} ({cf:.0%})")
        elif nm:
            out.append(nm)
    return out if out else ["None detected"]

def extract_read_text(result):
    rd = getattr(result, "read", None)
    if not rd: return "No text detected"
    blocks = unwrap_values(getattr(rd, "blocks", None))
    if not blocks: return "No text detected"
    lines_text = [ln.text for b in blocks for ln in unwrap_values(getattr(b, "lines", [])) if getattr(ln, "text", None)]
    return " | ".join(lines_text) if lines_text else "No text detected"


# ---------------------------
# CV Analyze
# ---------------------------
def analyze_image(image_bytes: bytes):
    if not cv_client:
        return {"error": "Computer Vision client not initialized"}
    result = cv_client.analyze(
        image_data=image_bytes,
        visual_features=[VisualFeatures.CAPTION, VisualFeatures.OBJECTS, VisualFeatures.TAGS, VisualFeatures.READ]
    )
    caption = getattr(result, "caption", None)
    desc = getattr(caption, "text", None) if caption else None
    conf = getattr(caption, "confidence", None) if caption else None
    return {
        "description": desc or "No description available",
        "confidence": f"{conf:.0%}" if isinstance(conf, (int, float)) else "N/A",
        "objects": extract_objects(result),
        "tags": extract_tags(result, limit=5),
        "text": extract_read_text(result),
    }


# ---------------------------
# Content Safety
# ---------------------------
def check_safety(text: str, threshold: int = 4):
    if not safety_client:
        return False, None, {"warning": "ContentSafety client not initialized"}
    try:
        resp = safety_client.analyze_text(AnalyzeTextOptions(text=text))
        severities, blocked, reason = {}, False, None
        for item in getattr(resp, "categories_analysis", []) or []:
            cat, sev = getattr(item, "category", None), getattr(item, "severity", None)
            if cat is not None and sev is not None:
                severities[str(cat)] = int(sev)
                if int(sev) >= threshold:
                    blocked, reason = True, str(cat)
        return blocked, reason, severities
    except Exception as e:
        return False, None, {"error": str(e)}


# ---------------------------
# Azure OpenAI Response
# ---------------------------
def generate_response(image_analysis: dict, user_question: str, user_id="default"):
    if not openai_client:
        return "❌ OpenAI client not initialized. Check AZURE_OPENAI_KEY."
    
    blocked, reason, _ = check_safety(user_question)
    if blocked:
        return f"⚠️ Blocked input: {reason}"
    
    if "error" in image_analysis:
        return f"❌ Image analysis failed: {image_analysis['error']}"

    context = (
        f"Image Analysis:\n"
        f"Description: {image_analysis['description']} (Confidence: {image_analysis['confidence']})\n"
        f"Objects: {', '.join(image_analysis['objects'])}\n"
        f"Tags: {', '.join([t.split(' (')[0] for t in image_analysis['tags']])}\n"
        f"Text in image: {image_analysis['text']}\n"
    )

    history = conversation_history.get(user_id, [])
    messages = [
        {"role": "system", "content": "You are a helpful multimodal assistant. Use ONLY the provided image analysis."},
        *history[-4:],
        {"role": "user", "content": f"{context}\nQuestion: {user_question}"}
    ]

    resp = openai_client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=messages,
        max_tokens=400,
        temperature=0.6
    )
    answer = resp.choices[0].message.content

    blocked_out, reason_out, _ = check_safety(answer)
    if blocked_out:
        return f"⚠️ Output blocked: {reason_out}"

    history.extend([{"role": "user", "content": user_question}, {"role": "assistant", "content": answer}])
    conversation_history[user_id] = history[-6:]
    return answer


# ---------------------------
# UI Widgets
# ---------------------------
def make_uploader():
    return widgets.FileUpload(accept="image/*", multiple=False, description="📤 Upload Image",
                              button_style="primary", layout=Layout(width="220px"))

image_upload = make_uploader()
question_text = widgets.Text(placeholder="Type question then click Analyze", description="❓ Question:",
                             layout=Layout(width="420px"))
submit_btn = widgets.Button(description="✨ Analyze", button_style="success", icon="comment",
                            layout=Layout(width="150px"))
refresh_btn = widgets.Button(description="🔄 Refresh", button_style="warning", icon="refresh",
                             layout=Layout(width="150px"))
preview_area = widgets.Output(layout=Layout(border="1px solid #e0e0e0", padding="10px", margin="10px 0"))
output_area = widgets.Output(layout=Layout(border="1px solid #e0e0e0", padding="15px", margin="10px 0"))

header = HBox([image_upload, question_text, submit_btn, refresh_btn],
              layout=Layout(margin="0 0 10px 0", align_items="center"))

ui = VBox([
    widgets.HTML("""
        <h2 style="color:#1976d2; margin:0">🖼️ Multimodal AI Assistant</h2>
        <ol style="color:#546e7a; margin:8px 0 15px 22px; font-size:0.95em; line-height:1.6">
          <li><b>Upload Image</b> – Preview appears immediately below.</li>
          <li><b>Enter Your Question</b> – Type what you want to know.</li>
          <li><b>Click Analyze</b> – AI processes image and responds.</li>
          <li><b>Click Refresh</b> – Reset to upload a new image.</li>
        </ol>"""),
    header,
    widgets.HTML("<b>Preview:</b>"), preview_area,
    widgets.HTML("<b>Results:</b>"), output_area
])


# ---------------------------
# Upload Handler
# ---------------------------
def on_upload_change(change):
    global current_image_bytes
    preview_area.clear_output()
    output_area.clear_output()
    val = image_upload.value
    if not val:
        current_image_bytes = None
        return
    fileinfo = next(iter(val.values())) if isinstance(val, dict) else (val[0] if isinstance(val, (list,tuple)) else val)
    content = fileinfo.get("content") if isinstance(fileinfo, dict) else getattr(fileinfo, "content", None)
    if content is None:
        current_image_bytes = None
        with preview_area:
            display(widgets.HTML("<span style='color:#d32f2f; font-weight:bold'>❌ Upload failed – no image data received</span>"))
        return
    current_image_bytes = content
    with preview_area:
        try:
            img = Image.open(io.BytesIO(current_image_bytes))
            img.thumbnail((700, 700))
            buf = io.BytesIO()
            img.save(buf, format="PNG")
            display(IPyImage(data=buf.getvalue()))
        except Exception as e:
            display(widgets.HTML(f"<span style='color:#d32f2f'>❌ Preview failed: {e}</span>"))

image_upload.observe(on_upload_change, names="value")


# ---------------------------
# Refresh Handler
# ---------------------------
def on_refresh(_):
    global current_image_bytes, image_upload
    preview_area.clear_output()
    output_area.clear_output()
    question_text.value = ""
    conversation_history.clear()
    current_image_bytes = None
    new_upload = make_uploader()
    new_upload.observe(on_upload_change, names="value")
    try: image_upload.close()
    except: pass
    image_upload = new_upload
    header.children = (image_upload, question_text, submit_btn, refresh_btn)
    with output_area:
        display(widgets.HTML("<span style='color:#388e3c; font-weight:bold'>✅ Reset complete. Upload a new image.</span>"))

refresh_btn.on_click(on_refresh)


# ---------------------------
# Analyze Handler
# ---------------------------
def on_submit(_):
    output_area.clear_output()
    if current_image_bytes is None:
        with output_area:
            display(widgets.HTML("<span style='color:#d32f2f; font-weight:bold'>❌ Upload an image first</span>"))
        return
    q = question_text.value.strip()
    if not q:
        with output_area:
            display(widgets.HTML("<span style='color:#d32f2f; font-weight:bold'>❌ Enter a question</span>"))
        return
    with output_area:
        display(widgets.HTML("<b style='color:#1976d2'>🔍 Running Computer Vision...</b>"))
        try:
            analysis = analyze_image(current_image_bytes)
        except Exception as e:
            display(widgets.HTML(f"<span style='color:#d32f2f'>❌ CV error: {e}</span>"))
            return
        if "error" not in analysis:
            display(widgets.HTML(
                f"<div style='background:#f8f9fa; padding:12px; border-radius:8px; margin:12px 0'>"
                f"<b>CV Output:</b><br>"
                f"• <b>Description:</b> {analysis['description']}<br>"
                f"• <b>Objects:</b> {', '.join(analysis['objects'])}<br>"
                f"• <b>Tags:</b> {', '.join(analysis['tags'])}<br>"
                f"• <b>Text:</b> {analysis['text']}"
                f"</div>"))
        display(widgets.HTML("<b style='color:#1976d2'>🧠 Generating AI response...</b>"))
        try:
            answer = generate_response(analysis, q)
            display(widgets.HTML(
                f"<div style='background:#e3f2fd; padding:16px; border-radius:10px; margin-top:10px'>"
                f"<b>Assistant:</b><br>{answer.replace(chr(10), '<br>')}"
                f"</div>"))
        except Exception as e:
            display(widgets.HTML(f"<span style='color:#d32f2f'>❌ OpenAI error: {e}</span>"))
    question_text.value = ""

submit_btn.on_click(on_submit)

# Display UI
display(ui)

✅ Azure services connected successfully
